In [1]:
# The notebook will need couple module files from code directory.
import os,sys
# os.path.join('..', 'code') is the relative path for code folder, module_path convert it to absolute path, final absolute url: d:\\Study\\Python\\llm-zoomcamp\\code
module_path = os.path.abspath(os.path.join('..', '02-vector-search'))
if module_path not in sys.path:
    sys.path.append(module_path)


## Q1. Embedding a query
Embed the following query:
> How does approximate nearest neighbor search work?
The embedder returns a vector of 384 numbers. What's the first value
(`v[0]`)?

In [2]:
from embedder import Embedder
embed=Embedder()

query="How does approximate nearest neighbor search work?"
v=embed.encode(query)
print(v[0])

-0.02058203437252893


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two
of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed
its `content`, and compute the cosine similarity with the query vector
from Q1. What do you get?

In [4]:
doc=[doc for doc in documents if doc['filename']=="02-vector-search/lessons/07-sqlitesearch-vector.md"][0]



In [5]:
vdoc=embed.encode(doc["content"])
#Q2
vdoc.dot(v)

np.float64(0.36107027225589694)

In [6]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [7]:
texts=[ doc["content"] for doc in chunks ]

In [8]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [9]:
scores = X.dot(v)

In [10]:
#3
chunks[np.argmax(scores)]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [11]:
from minsearch import VectorSearch

vindex=VectorSearch(keyword_fields=["content"])
vindex.fit(X,chunks)

In [12]:
q4="What metric do we use to evaluate a search engine?"
q4_vector=embed.encode(q4)

results = vindex.search(
    q4_vector, 
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [13]:
#Q4
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [14]:
#Q5
q5="How do I store vectors in PostgreSQL?"
q5_vector=embed.encode(q5)

results = vindex.search(
    q5_vector, 
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [15]:
#Q5 vector serach
[r["filename"] for r in results]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [23]:
from minsearch import Index

index=Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(chunks)

In [24]:
boost_dict={'content':3.0,'filename':1.0}
filter_dict={}

results_t=index.search(
    query=q5,
    num_results=5,
    boost_dict=boost_dict,
    filter_dict=filter_dict
)

In [25]:
#Q5 text serach
[r["filename"] for r in results_t]

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [26]:
#Q6 Hybrid search
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [27]:
q6="How do I give the model access to tools?"
q6_vector=embed.encode(q6)
vector_results= vindex.search(
    q6_vector, 
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

text_results=index.search(
    query=q6,
    num_results=5,
    boost_dict=boost_dict,
    filter_dict=filter_dict
)


In [29]:
results = rrf([vector_results, text_results])

In [30]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 